In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하길 권장해요.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
[Circuit의 명령어 시각화](/guides/visualize-circuits) 외에도, Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) 메서드를 사용하여 Circuit의 스케줄링을 시각화할 수 있습니다. 이 시각화는 예를 들어 Qubit의 유휴 시간을 빠르게 파악하는 데 도움이 될 수 있습니다. 다만, 이 메서드는 동적 Circuit에 대해 정확한 결과를 반환하지 않습니다. 동적 Circuit 스케줄링을 시각화하려면 [Qiskit Runtime 지원](#qr-support) 섹션에 설명된 `draw_circuit_schedule_timing` 메서드를 사용하세요.

## 예시

스케줄된 Circuit 프로그램을 시각화하려면, 일련의 제어 인수와 함께 이 함수를 호출할 수 있습니다. 출력 이미지의 대부분의 모양은 스타일시트로 수정할 수 있지만, 반드시 필요한 것은 아닙니다.

### 기본 스타일시트로 그리기

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### 프로그램 디버깅에 적합한 스타일시트로 그리기

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

사용자 정의 generator 또는 layout 함수를 만들고, 기존 스타일시트를 해당 함수로 업데이트할 수 있습니다. 이렇게 하면 스케줄된 Circuit 드로어의 코드베이스를 수정하지 않고도 출력 이미지의 대부분의 모양을 제어할 수 있습니다. 더 많은 예시는 [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) API 레퍼런스를 참조하세요.
<span id="qr-support"></span>
## Qiskit Runtime 지원
Qiskit에 내장된 타임라인 드로어는 정적 Circuit에 유용하지만, 브로드캐스팅 및 분기 결정과 같은 암묵적인 작업으로 인해 [동적 Circuit](/guides/classical-feedforward-and-control-flow)의 타이밍을 정확하게 반영하지 못할 수 있습니다. 동적 Circuit 지원의 일환으로, Qiskit Runtime은 요청 시 작업 결과 내에 정확한 Circuit 타이밍 정보를 반환합니다.

> **Note:** - 이것은 실험적인 함수입니다. 미리보기 출시 상태이므로 변경될 수 있습니다.
> - 이 함수는 Qiskit Runtime Sampler 작업에만 적용됩니다.
> - 총 Circuit 시간은 "compilation" 메타데이터에 반환되지만, 이는 청구에 사용되는 시간(양자 시간)이 아닙니다.
### 타이밍 데이터 검색 활성화
타이밍 데이터 검색을 활성화하려면, primitive 작업 실행 시 실험적 `scheduler_timing` 플래그를 `True`로 설정하세요.